# CS336 Spring 2026 - 10 RMSNorm / SwiGLU / Residual
----
> 09 学会了 `Linear` 和 `Embedding`。本章把这些 primitive 组合成 Transformer 的内部子层。

```text
x (B,T,D)
 ├─ RMSNorm → Attention/Linear → + x
 └─ RMSNorm → SwiGLU          → + residual
```

本章不先写完整 Transformer，而是逐个解释数值稳定性、张量维度与梯度流动。

## 1. 为什么需要 Normalization？

网络越深，中间激活的尺度可能变化很大，使训练不稳定。RMSNorm 通过最后一维的均方根进行缩放，并乘可训练 gain `g`：

$$
\mathrm{RMSNorm}(x)
= \frac{x}{\sqrt{\frac1D \sum_{i=1}^D x_i^2+\epsilon}}\odot g.
$$

与 LayerNorm 的重要差别：RMSNorm **不减均值**。Normalization 只发生在最后的 `D` 维，不跨 batch 或时间维。

In [1]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(336)
torch.set_num_threads(2)


class TeachingRMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # 在较高精度上计算平方和，降低低精度下的累积误差。
        xf = x.float()
        rms_inverse = torch.rsqrt(xf.square().mean(dim=-1, keepdim=True) + self.eps)
        normalized = (xf * rms_inverse).to(x.dtype)
        return normalized * self.weight.to(x.dtype)


B, T, D = 2, 4, 8
x = torch.randn(B, T, D)
norm = TeachingRMSNorm(D)
y = norm(x)

assert y.shape == x.shape
assert norm.weight.shape == (D,)

print("RMSNorm:", tuple(x.shape), "→", tuple(y.shape))

RMSNorm: (2, 4, 8) → (2, 4, 8)


## 2. RMSNorm 的数值实验

当 `epsilon=0` 且输入不为 0 时，`RMSNorm(c*x) ≈ RMSNorm(x)`（正的 `c`）；这说明它主要控制尺度。

真实实现需要 `epsilon>0`，否则全零向量会产生除零问题。低精度输入常将平方求和提升到 FP32，再将结果转回输入精度。


In [2]:
scale_invariant = TeachingRMSNorm(D, eps=0.0)
xx = torch.randn(2, 3, D) + 0.1
torch.testing.assert_close(
    scale_invariant(xx), scale_invariant(5.0 * xx), atol=1e-6, rtol=1e-6
)

zero_input = torch.zeros(B, T, D)

assert torch.isfinite(norm(zero_input)).all()
assert torch.all(norm(zero_input) == 0.0)

low_precision_input = torch.randn(2, 3, D).to(torch.bfloat16)
low_precision_output = norm(low_precision_input)

assert low_precision_output.dtype == torch.bfloat16
assert torch.isfinite(low_precision_output).all()

print("scale / zero / bfloat16: All tests passed!")

scale / zero / bfloat16: All tests passed!


## 3. SwiGLU 为什么使用三次 Linear？

传统 FFN 常写作：

$$
\mathrm{FFN}(x)=W_2\,\mathrm{ReLU}(W_1x).
$$

SwiGLU 增加 gating 分支：

$$
\mathrm{SwiGLU}(x)
= W_2\bigl(\mathrm{SiLU}(W_1x)\odot (W_3x)\bigr),
\qquad \mathrm{SiLU}(z)=z\,\sigma(z).
$$

shape：

```text
(B,T,D) → W1 → (B,T,F)
        → W3 → (B,T,F)
elementwise gate → (B,T,F)
        → W2 → (B,T,D)
```

参数量（无 bias）为 `3 * D * F`。


In [3]:
class TeachingSwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


F_DIM = 16
ffn = TeachingSwiGLU(D, F_DIM)
out = ffn(x)

assert out.shape == x.shape
assert sum(p.numel() for p in ffn.parameters()) == 3 * D * F_DIM

maual = ffn.w2((ffn.w1(x) * torch.sigmoid(ffn.w1(x))) * ffn.w3(x))
torch.testing.assert_close(out, maual)

print("SwiGLU numerical equivalence / shape / params checked: Passed")

SwiGLU numerical equivalence / shape / params checked: Passed


## 4. Residual Connection 的核心是 identity path

$$
y = x + f(x)
$$

反向传播时：

$$
\frac{\partial y}{\partial x} = I + \frac{\partial f(x)}{\partial x}.
$$

`I` 表示即使子层的梯度非常小，也有一条直接传递梯度的路径。它不意味着**所有**深网络都绝不会梯度问题，而是重要的稳定性机制。

使用残差时 `f(x)` 必须输出与 `x` 相同的 shape。


In [4]:
linear = nn.Linear(D, D, bias=False)
with torch.no_grad():
    linear.weight.zero_()

x0 = torch.randn(2, 3, D, requires_grad=True)
residual = x0 + linear(x0)
residual.sum().backward()

torch.testing.assert_close(x0.grad, torch.ones_like(x0))

print("当分支权重为零, identity path 的梯度为1")

当分支权重为零, identity path 的梯度为1


## 5. Pre-Norm Block 的顺序

对一个具有 attention 和 FFN 的 Transformer Block，Pre-Norm 常写作：

$$
h = x + \mathrm{Attention}(\mathrm{RMSNorm}(x))
$$

$$
y = h + \mathrm{SwiGLU}(\mathrm{RMSNorm}(h)).
$$

注意第二个 RMSNorm 作用在 `h` 而不是最初的 `x` 上。

先用一个形状保持的简化 sublayer 验证残差顺序；真正 attention 在 11–12 中实现。


In [5]:
class TeachingResidualBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.norm1 = TeachingRMSNorm(d_model)
        self.attention_placeholder = nn.Linear(d_model, d_model, bias=False)
        self.norm2 = TeachingRMSNorm(d_model)
        self.ffn = TeachingSwiGLU(d_model, d_ff)

    def forward(self, x):
        h = x + self.attention_placeholder(self.norm1(x))
        return h + self.ffn(self.norm2(h))


block = TeachingResidualBlock(D, F_DIM)
input_tensor = torch.randn(B, T, D, requires_grad=True)
block_output = block(input_tensor)

assert block_output.shape == input_tensor.shape

block_output.square().mean().backward()

assert torch.isfinite(input_tensor.grad).all()

print("Pre-Norm residual block: forward/backward 通过")

Pre-Norm residual block: forward/backward 通过


## 6. 重要的正确性检查

除了正常输入，还应测试：

- `T=1`：必须保留 batch 与时间维；
- `x=0`：RMSNorm 输出有限；
- 小幅和大幅输入缩放：输出不应无意义爆炸；
- 浮点精度：RMS 归一化的统计通常用 FP32 求和；
- `.backward()`：梯度要有限，Parameter 必须真正被注册。

本章验证的是**子层接口与数学行为**，还不是整个语言模型的正确性。


In [6]:
for T_case in (1, 2, 9):
    inp = torch.randn(3, T_case, D, requires_grad=True)
    result = block(inp)
    assert result.shape == (3, T_case, D)
    result.mean().backward()
    assert torch.isfinite(inp.grad).all()

registered = dict(block.named_parameters())

assert "norm1.weight" in registered
assert "ffn.w2.weight" in registered

print("单 token / 长序列 / 参数注册 / 梯度：通过")

单 token / 长序列 / 参数注册 / 梯度：通过


## 7. 小结 → 11 Scaled Dot-Product / Causal Attention

RMSNorm、SwiGLU 与 Residual 已经可以独立工作。下一步需要填入 Attention 子层，并理解：

```text
(B,T,D) → Q,K,V → (B,H,T,D_head)
          ↓ scores (B,H,T,T)
       causal mask + softmax
          ↓ weighted sum
        (B,T,D)
```

本章的所有子层都保持了可以被 Transformer Block 组合的输出 shape。